# Kp-Vsys posterior map

Shows the 2D posterior `P(Kp, vsys | data)` obtained by marginalising the logL
over the model amplitude alpha, with sigma contours and 1D marginals.

**Workflow:**
1. Load logl grid results
2. Compare integration methods (optional validation)
3. Compute alpha-marginalised posterior
4. Plot the Kp-Vsys map with sigma contours
5. Per-order maps (diagnostic)
6. Alpha marginal (is the model amplitude constrained?)

All heavy computation runs on the cluster via `run_starships_logl_grid`.
This notebook is for analysis only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import starships.logl_grid as lg
from starships.plotting_fcts import (
    plot_kpvsys_map,
    plot_logl_per_order,
    plot_alpha_marginal,
    plot_alpha_rv_map,
    find_isolated_peak_sigma,
)

## 1. Paths and parameters

In [ ]:
path_results = Path.home() / 'scratch/DataAnalysis/SPIRou/logl_grids/'

# Glob pattern — grid parameters are encoded in the filename:
#   {model_stem}_kp{min}_{max}_{step}_rv{min}_{max}_{step}[_noalpha]_visit*.npz
file_glob = 'take3_HRR_modif_disso_31254924_all_kp*_visit*.npz'

# For labelling / expected position
kp_ref      = 227.15   # km/s
rv_expected = 0.0      # km/s

## 2. Load grid results

In [ ]:
files = sorted(path_results.glob(file_glob))
print(f'Found {len(files)} file(s).')

lg.load_logl_results(files)

alpha_frac = lg._loaded_extra['alpha_frac']
# idx_signal: exposures where alpha_frac > 0.5 have detectable planet signal
# (in-transit for transmission spectroscopy, out-of-eclipse for emission).
(idx_signal,) = np.nonzero(alpha_frac > 0.5)
print(f'Exposures with planet signal: {len(idx_signal)}/{len(alpha_frac)}')

## 3. (Optional) Compare integration methods

The notebook used four methods to marginalise over alpha.  Below is a quick
sanity check that they agree (they should give identical contours).

In [ ]:
from scipy import integrate

alpha_array = np.linspace(0.01, 2., 31)
d_alpha = alpha_array[1] - alpha_array[0]

# Build the (n_alpha, n_vsys, n_kp) logL cube — vectorised over alpha
logl_map_alpha = lg.get_logl(
    idx_exposure=idx_signal, alpha=alpha_array, sum_axis=(-2, -1),
)
print(f'logl_map_alpha range: ({logl_map_alpha.min():.4g}, {logl_map_alpha.max():.4g})')

# Pre-normalise to max=0 before any exp() call.
if isinstance(logl_map_alpha, np.ma.MaskedArray):
    _shift = float(logl_map_alpha.max())
else:
    _finite = logl_map_alpha[np.isfinite(logl_map_alpha)]
    _shift = float(_finite.max()) if len(_finite) > 0 else 0.
logl_map_alpha = logl_map_alpha - _shift
print(f'After normalisation: ({logl_map_alpha.min():.4g}, {logl_map_alpha.max():.4g})')

# Method 1 (chosen): Simpson in log-space — _log_simps
marg_logsimp = np.exp(lg._log_simps(logl_map_alpha, d_alpha, axis=0))

# Method 2: Rectangular sum on exp(L)
exp_shift = np.nanmax(logl_map_alpha[np.isfinite(logl_map_alpha)])
marg_rect = np.sum(np.exp(logl_map_alpha - exp_shift), axis=0) * d_alpha * np.exp(exp_shift)

# Quick visual comparison along a vsys slice at the reference Kp
i_kp = int(np.argmin(np.abs(lg.kp_axis - kp_ref)))
print(f'Slice at Kp = {lg.kp_axis[i_kp]:.2f} km/s (requested {kp_ref:.2f})')

fig, ax = plt.subplots(figsize=(8, 3))
with np.errstate(divide='ignore'):
    ax.plot(lg.vsys_axis, np.log(marg_logsimp[:, i_kp]), label='log-Simpson (chosen)')
    ax.plot(lg.vsys_axis, np.log(marg_rect[:, i_kp]),    '--', label='rectangular sum')
ax.set_xlabel(r'$v_\mathrm{sys}$ (km/s)')
ax.set_ylabel(r'$\log P(K_P, v_\mathrm{sys} | \mathrm{data})$')
ax.set_title(f'Comparison at Kp = {lg.kp_axis[i_kp]:.1f} km/s')
ax.legend()
plt.tight_layout()

## 4. Compute alpha-marginalised posterior

`compute_kpvsys_posterior` handles:
- Looping over alpha values
- Log-domain Simpson integration (numerically stable)
- Oversampling the 2D posterior by cubic interpolation (for contour precision)
- Computing 1D marginals over Kp and vsys

In [ ]:
posterior, vsys_os, kp_os, margin_vsys, margin_kp = lg.compute_kpvsys_posterior(
    alpha_array=alpha_array,
    idx_signal=idx_signal,
    oversample=2,
    # kind='G',   # décommenter pour Gibson logL
)
print(f'Posterior shape (oversampled): {posterior.shape}')

## 5. Kp-Vsys map

Default: 1, 2, 3 sigma contours.  To use non-integer or custom sigma levels
(for example when the peak is very narrow), pass `sigma_levels=[2.5, 4.]`.

In [ ]:
%matplotlib inline

fig, axes = plot_kpvsys_map(
    posterior, vsys_os, kp_os, margin_vsys, margin_kp,
    n_sigma=3,
    # sigma_levels=[1., 2., 3.5],   # niveaux personnalisés
    sigma_display='contours',        # 'contours' | 'crosshairs' | 'none'
    # vsys_lim=(-30, 30),            # restreindre l'axe vsys (affichage + marginal)
    # kp_lim=(150, 300),             # restreindre l'axe Kp
    # crosshair_hole=0.05,           # taille du trou dans les crosshairs (mode 'crosshairs')
)

ax_map = axes[0]
ax_map.axvline(rv_expected, color='r', linestyle='--', alpha=0.5, label='vsys attendu')
ax_map.axhline(kp_ref,      color='r', linestyle='--', alpha=0.5)
ax_map.legend(fontsize=10)

# Pour sauvegarder :
# fig.savefig('kpvsys_map.pdf', bbox_inches='tight')

### 5.1 Peak position

In [ ]:
max_ind = np.unravel_index(np.argmax(posterior), posterior.shape)
vsys_peak = vsys_os[max_ind[0]]
kp_peak   = kp_os[max_ind[1]]
print(f'Peak: vsys = {vsys_peak:.2f} km/s,  Kp = {kp_peak:.2f} km/s')

### 5.2 Sigma maximum pour un pic isolé

`find_isolated_peak_sigma` retourne le sigma maximum pour lequel le contour ne contient encore qu'un seul pic (aucun pic secondaire absorbé). Utile quand la carte contient plusieurs candidats.

In [ ]:
sigma_isolated = find_isolated_peak_sigma(
    posterior, vsys_os, kp_os,
    sigma_max=10.,          # sigma maximum à scanner
    n_steps=200,            # pas de ~0.05σ → précision à 2 décimales
    min_secondary_pixels=3, # taille minimale d'un second pic pour le compter
)
print(f'Sigma max pour pic isolé : {sigma_isolated:.2f}σ')

# Afficher ce contour sur la carte
fig2, axes2 = plot_kpvsys_map(
    posterior, vsys_os, kp_os, margin_vsys, margin_kp,
    sigma_levels=[sigma_isolated],
    sigma_display='contours',
)
axes2[0].set_title(f'Contour isolé à {sigma_isolated:.2f}σ', fontsize=12)

# Ou laisser la fonction trouver toute seule :
# fig2, axes2 = plot_kpvsys_map(..., sigma_levels='auto')

## 6. Per-order diagnostic

Each panel is a full Kp-vsys map for one spectral order (summed over exposures).
A genuine detection appears consistently across orders at the same (vsys, Kp).
Orders with telluric contamination or low S/N will look noisy.

In [ ]:
# CCF per order: sum over exposures, keep order axis
# shape: (n_vsys, n_kp, n_orders)
ccf_orders = lg.get_ccf(idx_exposure=out_of_eclipse, sum_axis=-2, kind='BL')

# Mean number of valid pixels per order
n_valid = np.mean(lg.N, axis=0).astype(int)  # shape (n_exp, n_orders) → (n_orders,)
# If N is 1D (already meaned in load), adjust accordingly:
# n_valid = lg.N  if lg.N.ndim == 1 else np.mean(lg.N, axis=0)

fig, axes = plot_logl_per_order(
    ccf_orders, lg.vsys_axis, lg.kp_axis,
    n_valid=n_valid,
    n_col=6,
)
# To save:
# fig.savefig('per_order_ccf.pdf', bbox_inches='tight')

## 7. Alpha marginal

Shows how the logL (and the full marginal posterior) varies with the model
amplitude alpha.  A detection produces a peak near alpha ≈ 1.  A non-detection
gives a flat or monotonically declining posterior.

Two curves:
- **Black (slice)**: logL at the best-fit (vsys_peak, Kp_peak) as a function of alpha.
- **Blue (marginal)**: full marginal P(α|data), integrating over all (vsys, Kp).

In [ ]:
alpha_arr, logl_at_peak, log_p_alpha, vsys_pk, kp_pk = lg.compute_alpha_marginal(
    alpha_array=alpha_array,
    idx_signal=idx_signal,
)
print(f'Peak at vsys={vsys_pk:.2f} km/s, Kp={kp_pk:.2f} km/s')

fig, ax = plot_alpha_marginal(
    alpha_arr, logl_at_peak, log_p_alpha,
    vsys_peak=vsys_pk, kp_peak=kp_pk,
)
plt.tight_layout()
# fig.savefig('alpha_marginal.pdf', bbox_inches='tight')

## 8. Maps 2D alpha–Kp et alpha–vsys

Ces deux cartes donnent la distribution jointe entre l'amplitude du modèle α et
chacun des paramètres cinématiques, en marginalisant sur l'autre :

* **alpha–Kp** : `P(α, Kp | data) = ∫ L(α, vsys, Kp) d(vsys)` — est-ce que la détection
  est cohérente en amplitude pour toutes les valeurs de Kp ?
* **alpha–vsys** : `P(α, vsys | data) = ∫ L(α, vsys, Kp) d(Kp)` — idem pour vsys.

Une détection nette devrait montrer un pic près de (α ≈ 1, Kp_ref) et (α ≈ 1, vsys_ref).

In [ ]:
post_akp, alpha_os_akp, kp_os_akp, marg_alpha_akp, marg_kp_akp = \
    lg.compute_alpha_kp_posterior(
        alpha_array=alpha_array,
        idx_signal=idx_signal,
        oversample=2,
    )
print(f'Posterior alpha–Kp shape : {post_akp.shape}')

fig_akp, axes_akp = plot_alpha_rv_map(
    post_akp, alpha_os_akp, kp_os_akp, marg_alpha_akp, marg_kp_akp,
    rv_label=r'$K_{\rm P}$ (km s$^{-1}$)',
    n_sigma=3,
)
axes_akp[0].axhline(kp_ref, color='r', linestyle='--', alpha=0.5, label=r'$K_P$ ref')
axes_akp[0].legend(fontsize=9)
# fig_akp.savefig('alpha_kp_map.pdf', bbox_inches='tight')

In [ ]:
post_avs, alpha_os_avs, vsys_os_avs, marg_alpha_avs, marg_vsys_avs = \
    lg.compute_alpha_vsys_posterior(
        alpha_array=alpha_array,
        idx_signal=idx_signal,
        oversample=2,
    )
print(f'Posterior alpha–vsys shape : {post_avs.shape}')

fig_avs, axes_avs = plot_alpha_rv_map(
    post_avs, alpha_os_avs, vsys_os_avs, marg_alpha_avs, marg_vsys_avs,
    rv_label=r'$v_{\rm sys}$ (km s$^{-1}$)',
    n_sigma=3,
)
axes_avs[0].axhline(rv_expected, color='r', linestyle='--', alpha=0.5, label=r'$v_{\rm sys}$ ref')
axes_avs[0].legend(fontsize=9)
# fig_avs.savefig('alpha_vsys_map.pdf', bbox_inches='tight')

## 8. Save marginalised posterior for later

If you want to overlay multiple molecule combinations on the same Kp-Vsys plot,
save the marginalised posterior to an NPZ file and reload it in a comparison notebook.

In [ ]:
save_marg = False  # set to True to save

if save_marg:
    save_dir = Path('Figures/posterior_maps/')
    save_dir.mkdir(parents=True, exist_ok=True)
    save_file = save_dir / f'marg_kp_vsys_{file_stem}.npz'
    np.savez(
        save_file,
        posterior=posterior,
        vsys_axis=vsys_os,
        kp_axis=kp_os,
        margin_vsys=margin_vsys,
        margin_kp=margin_kp,
    )
    print(f'Saved: {save_file}')